# 03. Diffusion 계열 테스트셋 구축 (SDXL · FLUX1_dev · FLUX1_schnell)

ProGAN 얼굴 하나로만 학습한 모델이 **전혀 다른 생성 방식인 Diffusion까지 탐지하는지**
확인하기 위한 테스트셋을 만든 노트북.

- 출처: Kaggle `selfishgene/sfhq-t2i-synthetic-faces-from-text-2-image-models` (22.7GB, MIT)
- 모델별 4,000장씩 총 12,000장을 PNG로 변환 — **JPEG 재압축은 고주파 성분을 훼손해
  DWT·FFT 분석의 근거를 지우기 때문에** 무손실 PNG로 통일했다.
- 변환 병목을 `ThreadPoolExecutor`(8 workers) 병렬 처리로 해소
- 최종 평가에는 모델별 2,000장을 무작위 샘플링해 Real 2,000장과 1:1로 맞춤

> 원본 노트북에는 Kaggle 자격증명이 코드에 직접 들어 있었다. 공개하면서 환경변수
> 주입 방식으로 바꾸고 기존 키는 폐기했다.

In [ ]:
import os
import pandas as pd
from PIL import Image
from google.colab import drive
from tqdm import tqdm
import concurrent.futures

drive.mount('/content/drive')

# 자격증명은 코드에 쓰지 않고 환경변수/시크릿으로 주입
os.environ['KAGGLE_USERNAME'] = os.environ.get("KAGGLE_USERNAME")
os.environ['KAGGLE_KEY'] = os.environ.get("KAGGLE_KEY")

local_raw_path = "/content/sfhq_raw"
drive_final_path = "/content/drive/MyDrive/WayV_Detection_Test/3_fake_diffusion_png"
os.makedirs(local_raw_path, exist_ok=True)
os.makedirs(drive_final_path, exist_ok=True)

!kaggle datasets download -d selfishgene/sfhq-t2i-synthetic-faces-from-text-2-image-models --unzip -p "{local_raw_path}"

df = pd.read_csv(os.path.join(local_raw_path, "SFHQ_T2I_dataset.csv"))
target_models = ['SDXL', 'FLUX1_dev', 'FLUX1_schnell']

def convert_and_save(filename, model_folder):
    source = os.path.join(local_raw_path, "images", "images", filename)
    if os.path.exists(source):
        destination = os.path.join(model_folder, os.path.splitext(filename)[0] + ".png")
        try:
            with Image.open(source) as img:
                img.save(destination, "PNG")   # 고주파 보존을 위해 무손실 PNG
        except Exception:
            pass

for model in target_models:
    selected_df = df[df['model_used'] == model].head(4000)
    save_path = os.path.join(drive_final_path, model)
    os.makedirs(save_path, exist_ok=True)
    filenames = selected_df['image_filename'].tolist()

    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
        futures = {executor.submit(convert_and_save, f, save_path): f for f in filenames}
        for _ in tqdm(concurrent.futures.as_completed(futures), total=len(filenames)):
            pass

데이터셋 다운로드 중...
License(s): MIT
[SDXL] PNG 변환 및 저장 중... 4000/4000 [04:15<00:00]
[FLUX1_dev] PNG 변환 및 저장 중... 4000/4000 [03:59<00:00]
[FLUX1_schnell] PNG 변환 및 저장 중... 4000/4000 [04:05<00:00]
병렬 처리 변환 완료.

In [ ]:
import os

base_path = "/content/drive/MyDrive/WayV_Detection_Test/3_fake_diffusion_png"
target_models = ['SDXL', 'FLUX1_dev', 'FLUX1_schnell']

total_count = 0
for model in target_models:
    folder_path = os.path.join(base_path, model)
    if os.path.exists(folder_path):
        count = len([f for f in os.listdir(folder_path) if f.endswith('.png')])
        status = "확인 완료" if count == 4000 else f"목표치 4000장과 다름"
        print(f"[{model}] {count}장 — {status}")
        total_count += count
    else:
        print(f"[{model}] 폴더를 찾을 수 없음")

print("-" * 50)
print(f"총합계: {total_count}장 / 12000장")

[SDXL] 4000장 — 확인 완료
[FLUX1_dev] 4000장 — 확인 완료
[FLUX1_schnell] 4000장 — 확인 완료
--------------------------------------------------
총합계: 12000장 / 12000장

In [ ]:
import os, random, shutil, concurrent.futures
from google.colab import drive
from tqdm import tqdm

drive.mount('/content/drive', force_remount=True)

source_base_path = '/content/drive/MyDrive/WayV_Detection_Test'
target_base_path = '/content/drive/MyDrive'
target_models = ['FLUX1_dev', 'FLUX1_schnell', 'SDXL']
categories = ['0_real', '1_fake']
target_count = 2000     # Real:Fake = 1:1 로 맞춘다

def safe_copy(args):
    src, dst = args
    if not os.path.exists(dst):
        shutil.copy2(src, dst)     # 메타데이터까지 보존 복사

for model in target_models:
    for category in categories:
        source_dir = os.path.join(source_base_path, model, category)
        target_dir = os.path.join(target_base_path, model, category)
        os.makedirs(target_dir, exist_ok=True)

        all_files = [f for f in os.listdir(source_dir)
                     if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if len(all_files) >= target_count:
            # 특정 프롬프트/구도로 치우치지 않도록 무작위 샘플링
            selected = random.sample(all_files, target_count)
            tasks = [(os.path.join(source_dir, f), os.path.join(target_dir, f))
                     for f in selected]
            with concurrent.futures.ThreadPoolExecutor(max_workers=4) as ex:
                list(tqdm(ex.map(safe_copy, tasks), total=len(tasks),
                          desc=f"{model}_{category}", leave=False))

[FLUX1_dev] 0_real / 1_fake 2,000장 무작위 추출 및 복사 완료
[FLUX1_schnell] 0_real / 1_fake 2,000장 무작위 추출 및 복사 완료
[SDXL] 0_real / 1_fake 2,000장 무작위 추출 및 복사 완료
모든 복사 작업 완료.